# This is an explanation of the `api_eda.ipynb`
- a simple API client pattern. Meaning you wrap the HTTP request in a little helper, then build higher level functions for specific tasks. 
- Then you use a dunder guard for the entry point.
- Higher-level functions, like your example ones, define what you want. Helpers handle how. 
- That split is separation of concerns. You keep the messy HTTP details tucked away, so the top level reads like a step-by-step plan. 
- Clean, scalable, easy to test. That's the playbook.
- The script below Searches the JobTech API for job ads and prints the results in terminal. 
- It also makes a separate request to find out how many ads match your search.
- This script does not load anything into Snowflake.

## Import the tools

In [ ]:
import requests # lets Python send requests to the API
import json # json converts the APIs JSON response into Python obj dictionaries and lists

This builds the address `https://jobsearch.api.jobtechdev.se/search`
The f lets you insert a variable into a string using {url}.

In [ ]:
url = 'https://jobsearch.api.jobtechdev.se'
url_for_search = f"{url}/search"

## 2. Define the functions
Python reads these function definitions:
- Defining a function does not run its contents. It makes the function available to call later.

- Think of this as writing three sets of instructions before using them.

In [ ]:
def _get_ads(params):
def example_search_return_number_of_hits(query):
def example_search_loop_through_hits(query):

## 3. Start the actual search
- Execution reaches the bottom:
- The `if` means: run this block when this file is executed directly.

It then:

1. Stores the search text in query.
2. Calls the function that prints matching ads.
3. Calls the function that prints the total count.

Although the count function is defined first, the ads function is called first. The calling order determines what happens.

In [ ]:
if __name__ == '__main__':
    query = 'lärare uppsala'
    example_search_loop_through_hits(query)
    example_search_return_number_of_hits(query)

## 4. Prepare the request for job ads

- The first call is: `example_search_loop_through_hits(query)`

- The value 'lärare uppsala' enters the function as its query parameter:

| Setting | Meaning                     |
| ------- | --------------------------- |
| `q`     | The search text             |
| `limit` | Request up to this many ads |

In [ ]:
{
    'q': 'lärare uppsala',
    'limit': 100
}

In [ ]:
# This calls the helper function to fetch the data. The current function waits for the helper to return a result.
json_response = _get_ads(search_params) 

## 5. Send the request inside _get_ads

In [ ]:
def _get_ads(params):

- The dictionary named search_params in the calling function is now available as params inside this function.

- The names can differ. The dictionary is passed between them.


In [ ]:
headers = {'accept': 'application/json'}

- This tells the API: “Please return your response in JSON format.”
- It contains 3 strings:

| Part             | Purpose                                            |
| ---------------- | -------------------------------------------------- |
| `url_for_search` | Where to send the request                          |
| `headers`        | The requested response format                      |
| `params`         | What to search for and how many results to request |


In [ ]:
response = requests.get(
    url_for_search,
    headers=headers,
    params=params
)

- The API processes the search and sends back a response, stored in response.
- This checks for an HTTP error. If one occurs, it raises an exception. Because your code does not catch that exception, the script stops.

In [ ]:
response.raise_for_status()

## 6. Convert the response into Python data
- Read from the inside out:

| Expressions        | What it does                                  |
| ------------------ | --------------------------------------------- |
| `response.content` | Gets the response body as bytes               |
| `.decode('utf8')`  | Converts the bytes into text                  |
| `json.loads(...)`  | Converts JSON text into Python objects        |
| `return`           | Sends the result back to the calling function |


In [ ]:
return json.loads(response.content.decode('utf8'))

In [ ]:
# result
{
    "total": {
        "value": 245
    },
    "hits": [
        {
            "headline": "Teacher",
            "employer": {
                "name": "Example School"
            }
        },
        {
            "headline": "Math Teacher",
            "employer": {
                "name": "Another School"
            }
        }
    ]
}

Here:

- total contains information about the total number of matches.
- hits contains the returned job ads.
- Each ad is a dictionary.
- employer is another dictionary nested inside the ad.

## 7. Loop through the ads and print them

We return to this line with the fetched dictionary:
`json_response = _get_ads(search_params)`

Then:
`hits = json_response['hits']`

This selects only the list of ads.

In [ ]:
for hit in hits:
    print(f"{hit['headline']}, {hit['employer']['name']}")

- The loop takes one ad at a time and calls it hit.

- For each ad:

In [ ]:
hit['headline']

Gets the job title.

In [ ]:
hit['employer']['name']

- Gets the employer dictionary first, then its name.

- Using the example data, the output would be:

In [ ]:
Teacher, Example School
Math Teacher, Another School

# After the loop finishes, the function ends and execution returns to the bottom block.

## 8. Make a second request for the total count

The next call runs:

In [ ]:
example_search_return_number_of_hits(query)

Inside it:

In [ ]:
search_params = {'q': query, 'limit': 0}
json_response = _get_ads(search_params)

This calls the same helper again, but with limit: 0: request the count without individual ads.

In [ ]:
number_of_hits = json_response['total']['value']

This accesses two nested levels:

- Get the dictionary under total.
- Get the number under value.

In [ ]:
print(f"\nNumber of hits = {number_of_hits}")
# Number of hits = 245